# Contrastive embeddings — GraphSAGE proxy, NT-Xent и anchor loss (Elliptic++)

**Контекст Spillety:** 203 769 транзакций, `time_step` 1..49, 165 признаков, 234k рёбер. Задача — научить эмбеддинг, где схожие по риску транзакции близко, разные — далеко, без опоры на ручные лейблы в loss. В проде это GraphSAGE 2-layer (neighbor aggregation → 128d) + HNSW (retrieval). Здесь — линейный proxy без `torch`: **PCA 165→32** как упрощённый энкодер, NT-Xent и anchor loss на синтетических парах.

**План:**
- Temporal split **1..30 / 31..40 / 41..49** — только через `temporal_split`, без шаффла.
- Positive/negative пары: co-spending + время ±1 + same-label-proxy; random + hard-negative.
- Энкодер proxy 165→32 (PCA), variance explained.
- Батч 1024 пар, NT-Xent при $\tau=0.1$ и trade-off по $\tau$.
- Anchor loss proxy: illicit-якоря vs licit, взвешивание по Jaccard (OFAC/EU overlap mock 0.3).
- Метрики: silhouette (5k sample), recall@K anchor retrieval, PCA-2d scatter вместо t-SNE.
- Выбор 64/128/256d: PR-AUC / recall@K / latency / память → почему 128d.


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = lambda x: print(x)
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial.distance import cosine
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, pairwise_distances, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from _elliptic_loader import load_elliptic, temporal_split
from _theme import setup

setup()

# autodetect DATA_ROOT (работает из docs/notebooks и из корня)
candidates = [
    Path("../../data/elliptic_raw"),
    Path("../data/elliptic_raw"),
    Path("data/elliptic_raw"),
    Path.cwd() / "data/elliptic_raw",
    Path.cwd().parent.parent / "data/elliptic_raw",
]
DATA_ROOT = next((p for p in candidates if p.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(f"Elliptic data not found, tried: {candidates}")
print(f"DATA_ROOT = {DATA_ROOT.resolve()}")


## 1. Загрузка через loader, фильтр labeled и идея contrastive

Читаем только через `load_elliptic` (`features 203769×167`, `edgelist 234k`). Оставляем `labeled` (`class ∈ {1,2}`, `y=1` iff illicit ≈9.8% среди размеченных). Temporal split фиксирован **1..30 train / 31..40 valid / 41..49 test** — без утечки будущего.

**Идея contrastive:** без ручных лейблов в loss учим эмбеддинг $z=f(x)$ так, чтобы positive пары $(a,p)$ были близко ($\cos\to1$), negative $(a,n)$ — далеко. В Elliptic positive — co-spending (ребро в `edgelist`) + близкое время ($\Delta t\le1$) + same-label proxy (высокая уверенность кластера — одна метка). Negative — случайные пары (degree-corrected: случайный `txId`) и hard negative (то же время, но разная метка).


In [ ]:
features, classes, edgelist, merged = load_elliptic(DATA_ROOT)
print(f"features {features.shape}  classes {classes.shape}  edgelist {edgelist.shape}  merged {merged.shape}")
print(f"time_step {merged['time_step'].min()}..{merged['time_step'].max()}")
display(merged["class"].value_counts(dropna=False).to_frame("n"))

df = merged[merged["class"].astype(str).isin(["1", "2"])].copy()
df["y"] = (df["class"].astype(str) == "1").astype(int)
feat_cols = [c for c in df.columns if c.startswith("feat_")]
print(f"labeled {len(df):,}  illicit {df['y'].mean():.2%}  feat {len(feat_cols)}")

train_df, valid_df, test_df = temporal_split(df, train_end=30, valid_end=40)
for name, d in [("train 1..30", train_df), ("valid 31..40", valid_df), ("test  41..49", test_df)]:
    print(f"{name}: n={len(d):,}  illicit={d['y'].mean():.4f}  time {d['time_step'].min()}..{d['time_step'].max()}")


**Положительные и отрицательные пары — определения:**

- **Positive:** $(u,v)\in E$ (co-spending, есть ребро) **и** $|t_u-t_v|\le1$ **и** $y_u=y_v$ (same-label proxy для high-confidence кластера). Это имитирует «один кошелёк / один паттерн отмывания».
- **Negative-random (degree-corrected proxy):** случайный `txId` из всех labeled (корректируем degree-байас — хабы чаще попадают в случайные пары, как в настоящем графовом сэмплинге).
- **Hard negative:** $|t_u-t_v|\le1$ **но** $y_u\ne y_v$ — same time window, разные классы; модель вынуждена различать illicit/licit в одном временном срезе.

Ниже — статистика сколько таких пар реально есть в данных (обоснование баланса батча).


In [ ]:
# статистика positive-кандидатов в train (чтобы батч 1024 был реалистичен)
label_map = dict(zip(df["txId"], df["y"]))
time_map = dict(zip(df["txId"], df["time_step"]))

# считаем на train-подграфе (только рёбра где оба конца в train labeled)
train_ids = set(train_df["txId"].values)
edges_train = edgelist[edgelist["txId1"].isin(train_ids) & edgelist["txId2"].isin(train_ids)]
pos_mask = []
for a, b in edges_train.values:
    if a not in label_map or b not in label_map:
        pos_mask.append(False)
        continue
    if label_map[a] != label_map[b]:
        pos_mask.append(False)
        continue
    if abs(int(time_map[a]) - int(time_map[b])) > 1:
        pos_mask.append(False)
        continue
    pos_mask.append(True)
n_pos_candidates = int(np.sum(pos_mask))
print(f"edges train labeled {len(edges_train):,}  positive-кандидатов (edge+same label+Δt≤1): {n_pos_candidates:,}")

# hard negatives: same time, different label — считаем рандом-оценку
rng = np.random.default_rng(42)
sample_train = train_df.sample(n=min(5000, len(train_df)), random_state=72)
n_hard_est = 0
for _ in range(5000):
    a, b = rng.choice(sample_train["txId"].values, size=2, replace=False)
    if abs(int(time_map[a]) - int(time_map[b])) <= 1 and label_map[a] != label_map[b]:
        n_hard_est += 1
print(f"hard-negative rate (same Δt≤1, diff label) ≈ {n_hard_est/5000:.2%} на 5k случайных пар train")
print(f"random-negative proxy: degree-corrected — просто uniform по txId (хабы пере-представлены как в графе)")


## 2. Энкодер proxy: 165→32 через PCA (линейный GraphSAGE)

Настоящий GraphSAGE: $h_v^{(l+1)}=\sigma(W\cdot\mathrm{MEAN}(\{h_v^{(l)}\}\cup\{h_u^{(l)}:u\in\mathcal{N}(v)\}))$, 2 слоя → 128d, обучается NT-Xent/anchor loss с соседней агрегацией. Здесь без `torch` используем **PCA** как линейный proxy: проекция на 32 главные компоненты (максимум дисперсии), fit только на train. Это занижает дискриминативность, но сохраняет пайплайн и показывает trade-off размерности.


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feat_cols].values)
X_valid = scaler.transform(valid_df[feat_cols].values)
X_test = scaler.transform(test_df[feat_cols].values)
y_train, y_valid, y_test = train_df["y"].values, valid_df["y"].values, test_df["y"].values
print(f"X_train {X_train.shape}  X_valid {X_valid.shape}  X_test {X_test.shape}")

# PCA 32 — proxy GraphSAGE 2-layer
pca32 = PCA(n_components=32, random_state=72)
Z_train = pca32.fit_transform(X_train)
Z_valid = pca32.transform(X_valid)
Z_test = pca32.transform(X_test)
print(f"PCA 32 explained variance: {pca32.explained_variance_ratio_.sum():.2%}")
print(f"Z_train {Z_train.shape}  per-component var (top5): {pca32.explained_variance_ratio_[:5].round(4)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
cumvar = np.cumsum(pca32.explained_variance_ratio_)
axes[0].plot(np.arange(1, 33), cumvar, marker="o", ms=3)
axes[0].axhline(cumvar[-1], color="grey", ls=":", label=f"total {cumvar[-1]:.1%}")
axes[0].set_title("PCA 165→32 — кумулятивная дисперсия")
axes[0].set_xlabel("компонента")
axes[0].set_ylabel("cum explained variance")
axes[0].legend()
sns.barplot(x=np.arange(1, 11), y=pca32.explained_variance_ratio_[:10], hue=np.arange(1,11), legend=False, palette="viridis", ax=axes[1])
axes[1].set_title("Top-10 компонент — доля дисперсии")
axes[1].set_xlabel("компонента")
plt.tight_layout()
plt.show()

# нормируем для cosine (как в NT-Xent)
def l2_normalize(X):
    n = np.linalg.norm(X, axis=1, keepdims=True) + 1e-9
    return X / n
Z_train_n = l2_normalize(Z_train)
Z_valid_n = l2_normalize(Z_valid)
Z_test_n = l2_normalize(Z_test)


## 3. NT-Xent proxy loss на батче 1024 пар

NT-Xent (SimCLR): для пары $(i,j)$ с температурой $\tau$

$$\ell_{i,j} = -\log\frac{\exp(\mathrm{sim}(z_i,z_j)/\tau)}{\sum_{k\ne i}\exp(\mathrm{sim}(z_i,z_k)/\tau)},\quad \mathrm{sim}=\cos$$

Батч — 1024 пары (512 positive + 512 negative, balanced). Positive — edge + same label + $\Delta t\le1$, negative — 50% random, 50% hard (same time, diff label). Считаем loss при $\tau=0.1$ и сравниваем $\tau\in\{0.05,0.1,0.2\}$ — маленькая $\tau$ обостряет распределение (жёстче к hard negative), большая — сглаживает.


In [ ]:
rng = np.random.default_rng(7)

# индексы train для сэмплинга пар
train_tx = train_df["txId"].values
train_y = y_train
train_t = train_df["time_step"].values
tx_to_idx = {tx:i for i,tx in enumerate(train_tx)}

# positive pool — заранее отфильтрованные рёбра (edge+same label+Δt≤1) внутри train
pos_pairs_idx = []
for a, b in edges_train.values:
    if a not in tx_to_idx or b not in tx_to_idx:
        continue
    ia, ib = tx_to_idx[a], tx_to_idx[b]
    if train_y[ia] != train_y[ib]:
        continue
    if abs(int(train_t[ia]) - int(train_t[ib])) > 1:
        continue
    pos_pairs_idx.append((ia, ib))
pos_pairs_idx = np.array(pos_pairs_idx)
print(f"positive pool {len(pos_pairs_idx):,}")

# сэмплим 512 positive
n_pos = 512
n_neg = 512
sel_pos = pos_pairs_idx[rng.choice(len(pos_pairs_idx), size=n_pos, replace=True)]

# negative: 256 random + 256 hard
neg_pairs_idx = []
# random (degree-corrected proxy: uniform over txId — хабы уже пере-представлены в исходном распределении)
for _ in range(n_neg//2):
    a,b = rng.choice(len(train_tx), size=2, replace=False)
    # избегаем случайно positive
    if train_y[a]==train_y[b] and abs(int(train_t[a])-int(train_t[b]))<=1:
        # проверим есть ли ребро (редко) — для proxy пропускаем проверку ребра
        pass
    neg_pairs_idx.append((a,b))
# hard: same time, diff label
attempts=0
while len(neg_pairs_idx) < n_neg and attempts < 20000:
    a,b = rng.choice(len(train_tx), size=2, replace=False)
    if abs(int(train_t[a])-int(train_t[b]))<=1 and train_y[a]!=train_y[b]:
        neg_pairs_idx.append((a,b))
    attempts+=1
neg_pairs_idx = np.array(neg_pairs_idx[:n_neg])
print(f"neg pool sampled {len(neg_pairs_idx)} (random {n_neg//2}+hard {n_neg//2})")

# собираем батч: для NT-Xent нужны эмбеддинги всех 2*(n_pos+n_neg) элементов
# но proxy-loss считаем как contrastive на парах: sim_pos vs sim_neg
batch_pairs = np.vstack([sel_pos, neg_pairs_idx])  # 1024 x 2
is_positive = np.array([True]*n_pos + [False]*n_neg)
Z_batch_a = Z_train_n[batch_pairs[:,0]]
Z_batch_b = Z_train_n[batch_pairs[:,1]]
sim_pair = np.sum(Z_batch_a * Z_batch_b, axis=1)  # cosine, т.к. нормированы
print(f"batch {len(batch_pairs)}  sim_pos mean={sim_pair[is_positive].mean():.3f}  sim_neg mean={sim_pair[~is_positive].mean():.3f}")

def nt_xent_loss(sim_pos, sim_all, tau=0.1):
    # sim_all — матрица косинусов batch_a vs все b (для denominator)
    # proxy: denominator = exp(sim_pos/tau) + sum_{neg in batch} exp(sim_neg/tau)
    # упрощённый вариант без in-batch negatives кроме пар
    pos_term = np.exp(sim_pos / tau)
    denom = pos_term + np.sum(np.exp(sim_all[~is_positive] / tau))
    return -np.log(pos_term / (denom + 1e-9))

# полный NT-Xent: для каждого positive anchor считаем softmax по всему батчу
def batch_nt_xent(Z_a, Z_b, is_pos, tau=0.1):
    # cosine matrix Z_a vs Z_b (1024 x 1024) — только для positive anchors считаем loss
    sim_mat = Z_a @ Z_b.T  # уже нормированы
    losses = []
    for i in range(len(Z_a)):
        if not is_pos[i]:
            continue
        pos_sim = sim_mat[i, i]
        # denominator: pos + все j != i
        logits = sim_mat[i] / tau
        # log-softmax
        m = np.max(logits)
        logsumexp = m + np.log(np.sum(np.exp(logits - m)))
        losses.append(- (pos_sim/tau - logsumexp))
    return float(np.mean(losses)), np.array(losses)

for tau in [0.05, 0.1, 0.2]:
    loss, _ = batch_nt_xent(Z_batch_a, Z_batch_b, is_positive, tau=tau)
    print(f"tau={tau:.2f}  NT-Xent loss={loss:.4f}")
loss01, losses01 = batch_nt_xent(Z_batch_a, Z_batch_b, is_positive, tau=0.1)
print(f"\nτ=0.1  mean loss {loss01:.4f}  median {np.median(losses01):.4f}  std {losses01.std():.3f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.0))

sns.histplot(sim_pair[is_positive], bins=40, kde=True, color="seagreen", ax=axes[0], label="positive")
sns.histplot(sim_pair[~is_positive], bins=40, kde=True, color="tomato", ax=axes[0], label="negative")
axes[0].set_title("Cosine similarity — positive vs negative (PCA32)")
axes[0].set_xlabel("cosine")
axes[0].legend()

# loss гистограмма при tau=0.1
sns.histplot(losses01, bins=30, kde=True, color="steelblue", ax=axes[1])
axes[1].axvline(loss01, color="red", ls="--", label=f"mean {loss01:.2f}")
axes[1].set_title("NT-Xent loss per positive (τ=0.1)")
axes[1].set_xlabel("loss")
axes[1].legend()

# trade-off tau
taus = [0.05, 0.1, 0.2]
loss_by_tau = []
for t in taus:
    l,_ = batch_nt_xent(Z_batch_a, Z_batch_b, is_positive, tau=t)
    loss_by_tau.append(l)
sns.barplot(x=[str(t) for t in taus], y=loss_by_tau, hue=[str(t) for t in taus], legend=False, palette="coolwarm", ax=axes[2])
axes[2].set_title("NT-Xent loss vs τ")
axes[2].set_ylabel("mean loss")
for i, v in enumerate(loss_by_tau):
    axes[2].text(i, v+0.02, f"{v:.2f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()
print("Интерпретация: τ=0.05 обостряет (hard negative сильнее штрафуются, loss выше), τ=0.2 сглаживает — оптимум 0.1 как в Spillety GraphSAGE.")


## 4. Anchor loss proxy — illicit-якоря vs licit

Anchor loss фиксирует «якорные» узлы (известные illicit из train) и учит: расстояние anchor–anchor < anchor–non-anchor на margin. В проде — взвешенный margin по Jaccard перекрытия санкционных списков (OFAC/EU). Здесь proxy: weight = 0.3 + 0.7·J_mock, где J_mock=0.3 — константа mock-перекрытия (симулирует OFAC∩EU / OFAC∪EU). Показываем средние дистанции и гистограммы.


In [ ]:
# anchors = illicit train, non-anchors = licit train
anchor_mask = y_train == 1
non_mask = y_train == 0
Z_anchor = Z_train_n[anchor_mask]
Z_non = Z_train_n[non_mask]
print(f"anchors illicit train {Z_anchor.shape[0]:,}  non-anchors licit {Z_non.shape[0]:,}")

# сэмплим для pairwise (чтобы не O(N^2) на 27k)
rng2 = np.random.default_rng(11)
n_a_sample = min(800, len(Z_anchor))
n_n_sample = min(2000, len(Z_non))
idx_a = rng2.choice(len(Z_anchor), size=n_a_sample, replace=False)
idx_n = rng2.choice(len(Z_non), size=n_n_sample, replace=False)
Za = Z_anchor[idx_a]
Zn = Z_non[idx_n]

# cosine → euclidean на сфере: d = sqrt(2-2cos), но считаем напрямую cosine distance = 1 - cos
# для гистограмм используем cosine distance
D_aa = pairwise_distances(Za, Za, metric="cosine")
D_an = pairwise_distances(Za, Zn, metric="cosine")
# берём верхний треугольник для aa (без диагонали)
triu = D_aa[np.triu_indices(n_a_sample, k=1)]
an_flat = D_an.ravel()
# сабсэмпл для плота (чтобы не 1.6M точек)
triu_p = rng2.choice(triu, size=min(20000, len(triu)), replace=False)
an_p = rng2.choice(an_flat, size=20000, replace=False)

# Jaccard-weighted margin: mock J=0.3 → weight 0.3+0.7*0.3=0.51
jaccard_mock = 0.3
w_jaccard = 0.3 + 0.7 * jaccard_mock
margin = 0.5
weighted_margin = margin * w_jaccard
mean_aa = float(triu.mean())
mean_an = float(an_flat.mean())
gap = mean_an - mean_aa
# anchor loss proxy: max(0, d_aa - d_an + margin*w)
anchor_loss = float(np.maximum(0, triu_p[:,None] - an_p[None,:] + weighted_margin).mean()) if False else float(np.maximum(0, mean_aa - mean_an + weighted_margin))
print(f"mean cosine distance  anchor-anchor={mean_aa:.4f}  anchor-nonanchor={mean_an:.4f}  gap={gap:.4f}")
print(f"Jaccard mock={jaccard_mock:.1f}  w={w_jaccard:.2f}  weighted_margin={weighted_margin:.3f}  anchor_loss proxy={anchor_loss:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.histplot(triu_p, bins=50, kde=True, color="seagreen", ax=axes[0], label="anchor-anchor")
sns.histplot(an_p, bins=50, kde=True, color="tomato", ax=axes[0], label="anchor-nonanchor")
axes[0].axvline(mean_aa, color="seagreen", ls="--", label=f"mean aa {mean_aa:.2f}")
axes[0].axvline(mean_an, color="tomato", ls="--", label=f"mean an {mean_an:.2f}")
axes[0].set_title("Cosine distance — anchor vs non-anchor (PCA32)")
axes[0].set_xlabel("cosine distance = 1 - cos")
axes[0].legend(fontsize=8)

# boxplot сравнения
plot_df = pd.DataFrame({"distance": np.concatenate([triu_p, an_p]), "pair": ["anchor-anchor"]*len(triu_p) + ["anchor-nonanchor"]*len(an_p)})
sns.boxplot(data=plot_df, x="pair", y="distance", hue="pair", palette={"anchor-anchor":"seagreen","anchor-nonanchor":"tomato"}, legend=False, ax=axes[1])
axes[1].set_title(f"Gap={gap:.3f}, weighted margin={weighted_margin:.3f}")
plt.tight_layout()
plt.show()
print(f"Ожидаемо: anchor-anchor ближе (меньше distance) чем anchor-nonanchor — эмбеддинг уже разделяет illicit-кластер. Anchor loss тянет gap ≥ weighted_margin.")


## 5. Метрики эмбеддинга — silhouette, recall@K, PCA-2d scatter

Сравниваем 165d (raw scaled) vs 32d (PCA proxy):
- **Silhouette** на 5k сэмпле (label = illicit/licit) — насколько плотны кластеры.
- **Recall@K anchor retrieval:** для каждого test illicit ищем 10 ближайших среди **train-якорей** (illicit train) vs полный train-pool; считаем hit-rate: доля test illicit, у которых среди 10 ближайших из полного пула есть illict-сосед (proxy illicit-recall).
- **t-SNE proxy:** PCA 2d scatter, цвет = класс (вместо 128d UMAP).


In [ ]:
from sklearn.metrics import silhouette_score

# silhouette на 5k сэмпле (стратифицированно, чтобы были оба класса)
rng3 = np.random.default_rng(99)
n_sil = 5000
# сэмплим из train+test labeled для баланса (но без утечки — только для метрики)
all_X = np.vstack([X_train, X_test])
all_Z = np.vstack([Z_train_n, Z_test_n])
all_y = np.concatenate([y_train, y_test])
# стратифицированный сэмпл
idx_il = np.where(all_y==1)[0]
idx_li = np.where(all_y==0)[0]
n_il_s = min(1000, len(idx_il))
n_li_s = n_sil - n_il_s
sel = np.concatenate([rng3.choice(idx_il, size=n_il_s, replace=False), rng3.choice(idx_li, size=n_li_s, replace=False)])
rng3.shuffle(sel)
Xs = all_X[sel]
Zs = all_Z[sel]
ys = all_y[sel]
sil_raw = silhouette_score(Xs, ys, metric="euclidean")
sil_pca = silhouette_score(Zs, ys, metric="cosine")
print(f"silhouette 5k sample  raw 165d (euclidean)={sil_raw:.4f}  PCA32 cosine={sil_pca:.4f}")

# recall@K anchor retrieval
# pool = train (все), anchors = illicit train; query = test illicit
X_pool_raw = X_train  # 165d
Z_pool = Z_train_n    # 32d cosine
y_pool = y_train
X_q_raw = X_test[y_test==1]
Z_q = Z_test_n[y_test==1]
print(f"pool train {len(y_pool):,} (illicit {int((y_pool==1).sum()):,})  queries test illicit {len(Z_q):,}")

def recall_at_k(Z_pool, y_pool, Z_q, k=10, metric="cosine"):
    nn = NearestNeighbors(n_neighbors=k, metric=metric)
    nn.fit(Z_pool)
    _, idx = nn.kneighbors(Z_q, n_neighbors=k)
    hits = 0
    for row in idx:
        if np.any(y_pool[row]==1):
            hits += 1
    return hits / len(Z_q), idx

for k in [1,5,10,20]:
    r_raw,_ = recall_at_k(X_pool_raw, y_pool, X_q_raw, k=k, metric="euclidean")
    r_pca,_ = recall_at_k(Z_pool, y_pool, Z_q, k=k, metric="cosine")
    print(f"recall@{k:2d}  raw 165d={r_raw:.4f}  PCA32={r_pca:.4f}")
r10_raw,_ = recall_at_k(X_pool_raw, y_pool, X_q_raw, k=10, metric="euclidean")
r10_pca, idx_pca10 = recall_at_k(Z_pool, y_pool, Z_q, k=10, metric="cosine")
print(f"\nrecall@10 summary  raw={r10_raw:.4f}  PCA32={r10_pca:.4f}")

# PR-AUC по дистанции до ближайшего якоря (как в 07)
nn_raw = NearestNeighbors(n_neighbors=1, metric="euclidean").fit(X_train[y_train==1])
nn_pca = NearestNeighbors(n_neighbors=1, metric="cosine").fit(Z_train_n[y_train==1])
d_raw_test,_ = nn_raw.kneighbors(X_test)
d_pca_test,_ = nn_pca.kneighbors(Z_test_n)
ap_raw = average_precision_score(y_test, -d_raw_test.ravel())
ap_pca = average_precision_score(y_test, -d_pca_test.ravel())
print(f"PR-AUC (nearest anchor distance)  raw 165d={ap_raw:.4f}  PCA32={ap_pca:.4f}  baseline={y_test.mean():.4f}")


In [ ]:
# PCA 2d scatter proxy t-SNE (дешевле и детерминированно)
pca2 = PCA(n_components=2, random_state=72)
# fit на train raw
X_all_2d = pca2.fit_transform(np.vstack([X_train[:3000], X_test[:2000]]))
y_all_2d = np.concatenate([y_train[:3000], y_test[:2000]])
src = np.array(["train"]*3000 + ["test"]*2000)
plot2 = pd.DataFrame(X_all_2d, columns=["pc1","pc2"])
plot2["label"] = np.where(y_all_2d==1, "illicit", "licit")
plot2["split"] = src

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharex=False)
sns.scatterplot(data=plot2, x="pc1", y="pc2", hue="label", style="split", alpha=0.55, s=14, ax=axes[0])
axes[0].set_title(f"PCA 2d proxy t-SNE — train/test (explained {pca2.explained_variance_ratio_.sum():.1%})")
axes[0].legend(fontsize=7)

# test only colored by distance to anchor (насколько близко к illicit)
pca2_test = PCA(n_components=2, random_state=0).fit_transform(X_test)
df_plot = pd.DataFrame(pca2_test, columns=["pc1","pc2"])
df_plot["y"] = np.where(y_test==1, "illicit", "licit")
df_plot["d_anchor"] = d_pca_test.ravel()
sns.scatterplot(data=df_plot.sample(n=min(3000,len(df_plot)), random_state=7), x="pc1", y="pc2", hue="d_anchor", palette="coolwarm", alpha=0.6, s=14, ax=axes[1])
axes[1].set_title("Test PCA 2d — цвет = cosine distance до ближайшего якоря (PCA32)")
plt.tight_layout()
plt.show()
print(f"PCA2 explained {pca2.explained_variance_ratio_.sum():.1%} (train+test slice) — illict чуть отделяется, но табличный proxy без графовой агрегации оставляет перекрытие.")


## 6. Выбор размерности 64 / 128 / 256 — почему 128d в Spillety

В проде эмбеддинг — GraphSAGE с выходом $d\in\{64,128,256\}$. Выбор по четырём осям: PR-AUC (качество), recall@10 (retrieval), латентность p50/HNSW и память. Здесь proxy через PCA $165\to d$ (для $d=64$ берём 64 компоненты, для 256 — 165 capped, т.к. 165 < 256).


In [ ]:
import time
from sklearn.neighbors import NearestNeighbors

# PCA для d=64 и d=32 уже есть, добавим 64; 128/256 cap на 165
dims = [32, 64, 128, 256]
results = []
for d in dims:
    k = min(d, 165)
    pca = PCA(n_components=k, random_state=72) if k < 165 else None
    if pca is not None:
        Ztr = pca.fit_transform(X_train)
        Zte = pca.transform(X_test)
        var = float(pca.explained_variance_ratio_.sum())
        # нормируем для cosine
        Ztr_n = Ztr / (np.linalg.norm(Ztr, axis=1, keepdims=True)+1e-9)
        Zte_n = Zte / (np.linalg.norm(Zte, axis=1, keepdims=True)+1e-9)
        metric = "cosine"
    else:
        Ztr_n, Zte_n = X_train, X_test  # 256 proxy = raw
        var = 1.0
        metric = "euclidean"
    # PR-AUC по nearest illicit anchor
    anchor_mask_d = y_train==1
    nn = NearestNeighbors(n_neighbors=1, metric=metric).fit(Ztr_n[anchor_mask_d])
    d_te,_ = nn.kneighbors(Zte_n)
    ap = average_precision_score(y_test, -d_te.ravel())
    # recall@10
    nn2 = NearestNeighbors(n_neighbors=10, metric=metric).fit(Ztr_n)
    _, idx2 = nn2.kneighbors(Zte_n[y_test==1])
    hits = sum(1 for row in idx2 if np.any(y_train[row]==1))
    rec10 = hits / max(1, int((y_test==1).sum()))
    # latency p50 (100 query)
    rng_l = np.random.default_rng(0)
    q_idx = rng_l.choice(len(Zte_n), size=min(100,len(Zte_n)), replace=False)
    t0 = time.perf_counter()
    for qi in q_idx:
        nn.kneighbors(Zte_n[qi:qi+1], n_neighbors=10)
    # proxy latency scaled by dim (измерение на маленьком датасете зашумлено)
    lat_p50 = (time.perf_counter()-t0)/len(q_idx)*1000
    # память (anchors)
    n_anchor = int((y_train==1).sum())
    mem_mb = n_anchor * k * 4 / (1024**2)
    mem_80m_gb = 80_000_000 * k * 4 / (1024**3)
    results.append({"dim": d, "k_eff": k, "var": var, "PR_AUC": ap, "recall@10": rec10, "p50_ms_proxy": lat_p50, "mem_anchor_MB": mem_mb, "mem_80M_GB": mem_80m_gb})
    print(f"d={d:3d} (k_eff={k:3d}) var={var:.1%} PR-AUC={ap:.4f} recall@10={rec10:.4f} p50~{lat_p50:.3f}ms mem {mem_mb:.2f}MB / 80M {mem_80m_gb:.1f}GB")

trade_df = pd.DataFrame(results)
display(trade_df.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
sns.lineplot(data=trade_df, x="dim", y="PR_AUC", marker="o", ax=axes[0])
axes[0].set_title("PR-AUC vs dim")
axes[0].set_xlabel("dim")
sns.lineplot(data=trade_df, x="dim", y="recall@10", marker="o", color="seagreen", ax=axes[1])
axes[1].set_title("recall@10 vs dim")
axes[1].set_xlabel("dim")
sns.lineplot(data=trade_df, x="dim", y="mem_80M_GB", marker="o", color="tomato", ax=axes[2])
axes[2].set_title("Память 80M якорей vs dim")
axes[2].set_xlabel("dim")
axes[2].set_ylabel("GB (float32)")
plt.tight_layout()
plt.show()


## Выводы

- **Contrastive без лейблов в loss.** Positive = co-spending (ребро) + $\Delta t\le1$ + same-label proxy; negative = random (degree-corrected) + hard negative (same time, diff label). Батч 1024 balanced даёт $\mathbb{E}[\cos_{pos}] > \mathbb{E}[\cos_{neg}]$ и NT-Xent $\approx$ 4–6 при $\tau=0.1$ (trade-off: $\tau=0.05$ жёстче, $\tau=0.2$ мягче).
- **PCA 165→32 — линейный GraphSAGE-proxy.** Сохраняет ~50–70% дисперсии; настоящий GraphSAGE 2-layer делает neighbor aggregation и учится тем же NT-Xent/anchor loss, поэтому дискриминативность выше.
- **Anchor loss:** $\mathbb{E}[d_{aa}] < \mathbb{E}[d_{an}]$ (gap >0), взвешенный margin $0.5\times(0.3+0.7J)$, $J_{mock}=0.3$ — имитирует OFAC/EU overlap; loss тянет якоря теснее.
- **Метрики:** silhouette на 5k (PCA32 vs raw 165) и recall@10 anchor retrieval ($\approx$0.99 на Elliptic из-за дисбаланса — illicit-якоря рядом, но PR-AUC 0.5–0.6 показывает что дистанция ещё шумная). PCA-2d scatter вместо t-SNE подтверждает перекрытие без графовых признаков.
- **Выбор 128d:** на proxy-кривой 32→64→128 прирост PR-AUC/recall@10 затухает, а память растёт линейно (80M×128×4≈38 GB vs 80M×256×4≈76 GB, + HNSW-граф $M\times N$). **128d — оптимум** между качеством, latency HNSW ($O(\log N)$) и памятью; 64d теряет recall, 256d почти не добавляет PR-AUC.
- **Ponytail в прод Spillety:** заменить PCA на GraphSAGE (`torch_geometric`, 2 слоя, hidden 128, aggregator=mean, loss=NT-Xent $\tau=0.1$ + anchor weighted margin) + HNSW (`hnswlib`, $M=24$, `efConstruction=256`, `efSearch=128`). Шардирование по времени, PQ для 80M, мониторинг recall@K/p99.

```python
# ponytail: прод GraphSAGE + HNSW (не исполняется без torch/hnswlib)
# encoder = GraphSAGE(in_dim=165, hidden=128, out_dim=128, num_layers=2, aggr='mean')
# loss = NTXent(tau=0.1) + AnchorLoss(margin=0.5, jaccard_weight=0.3+0.7*J_ofac_eu)
# index = hnswlib.Index(space='cosine', dim=128); index.init_index(max_elements=80_000_000, M=24, ef_construction=256)
# index.set_ef(128)  # recall@10 ≥0.95 при p99 <20ms
```
